# Train road-pothole-seg on Colab

Thin launcher: all training logic is in `training/scripts/train.py`, settings in
`configs/train.yaml`. Set the options below, then **Runtime → Run all**.

Before the first run, upload `merged-v1-base.zip` to `MyDrive/road-pothole-seg/data/`.
Use a GPU runtime (**Runtime → Change runtime type → T4 GPU** or better).

In [ ]:
PROFILE = "sanity"  # sanity | teacher | baseline | small | tune1 (see configs/train.yaml)
DATASET = "merged-v1"  # zip name in DRIVE_DIR/data, without .zip
RUN_NAME = ""  # empty = profile name; set a new name to keep old runs
RESUME = False  # True = continue the run from its last.pt after a disconnect
REFERENCE = "baseline"  # finished run to compare against in the live panel; "" = none
DRIVE_DIR = "/content/drive/MyDrive/road-pothole-seg"

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

In [ ]:
# Get the latest code from GitHub (code reaches Colab only through git).
import os

if os.path.isdir("/content/road-pothole-seg"):
    !git -C /content/road-pothole-seg pull --ff-only
else:
    !git clone https://github.com/ngtdat9/road-pothole-seg.git /content/road-pothole-seg
%cd /content/road-pothole-seg
!git log --oneline -1
!pip install -q -e ".[train]"

In [ ]:
# Copy the dataset to the local disk: training from the Drive mount is very slow.
if not os.path.isdir(f"/content/{DATASET}"):
    !cp "{DRIVE_DIR}/data/{DATASET}.zip" /content/
    !unzip -q "/content/{DATASET}.zip" -d /content/
!wc -l /content/{DATASET}/*.txt

In [ ]:
# Train inside this notebook so the live panel (progress, ETA, mAP, losses) draws below;
# Ultralytics' own log follows it. Every epoch is also saved to runs/<run>/epochs.csv.
import sys
from pathlib import Path

for mod in [m for m in sys.modules if m.startswith("training")]:
    del sys.modules[mod]  # use the code just pulled, even when re-running this cell
from training.scripts.monitor import Monitor, load_reference  # noqa: E402
from training.scripts.train import git_commit, results_row, run  # noqa: E402

reference = load_reference(REFERENCE) if REFERENCE else None
monitor = Monitor(show=Monitor.notebook(), reference=reference, commit=git_commit())
summary = run(
    PROFILE,
    Path(f"/content/{DATASET}"),
    Path(f"{DRIVE_DIR}/runs"),
    name=RUN_NAME or None,
    resume=RESUME,
    callbacks=monitor.callbacks(),
)
print(summary["metrics"])
print("\nRow for docs/results.md (fill in the conclusion):")
print(results_row(summary))

Weights, curves and `summary.json` are in `MyDrive/road-pothole-seg/runs/<run name>/`.
Copy the printed results row into `docs/results.md` and add a one-line conclusion.